# AgriTech Decision Benchmark — Decision-1 vs. TypeSafe Jev

This notebook compares Google Decision-1 and TypeSafe Jev on 40 benchmark images and four isolated primitives: Boolean, categorical Choice, severity Score, and Spatial/Box. Predictions, accuracy, latency, token counts, HTTP status, and cost are checkpointed to the CLEAN6 audit workbook.

## Before running
In Google Colab, mount Drive, set the environment variable `AGRITECH_BASE_DIR` to the benchmark project folder, and add `GEMINI_API_KEY` and `OPENROUTER_API_KEY` under **Secrets** (not in notebook code). Enable notebook access to both secrets. No credential or credential prefix is printed.

## Cells
1. Project overview and run instructions.
2. Mount Drive; verify the CLEAN6 paths, all 40 image files, the workbook tabs, and securely load both secrets.
3. Full end-to-end prediction, accuracy, and telemetry runner for all four primitives. Set `LATENCY_AND_SPATIAL_CONF_ONLY = False` to re-run predictions and telemetry.
4. Configurable range telemetry runner, defaulting to spatial-confidence/latency/telemetry-only updates so existing verified predictions and accuracy are preserved. Both runners checkpoint each image to the workbook; review range, model availability, and provider pricing before execution.

Run cells 1–2 first. Review model pricing and image ranges before running either benchmark cell. API calls incur provider-specific charges/quotas; run only in an authorized Colab environment.

In [ ]:
# Setup, Environment & Workbook Verification
import os
import pandas as pd
import openpyxl
from google.colab import drive, userdata

if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount('/content/drive')

BASE_DIR = os.environ.get("AGRITECH_BASE_DIR")
if not BASE_DIR:
    raise RuntimeError("Set AGRITECH_BASE_DIR to the benchmark project directory before running this cell.")
EXCEL_PATH = f"{BASE_DIR}/Project_AgriTech_Decision_Audit_CLEAN6.xlsx"
GROUND_TRUTH_PATH = f"{BASE_DIR}/ground_truth.csv"
RAW_LOGS_DIR = f"{BASE_DIR}/raw_api_logs"

assert os.path.isdir(BASE_DIR), f"Project directory not found: {BASE_DIR}"
assert os.path.isfile(GROUND_TRUTH_PATH), f"Ground-truth CSV not found: {GROUND_TRUTH_PATH}"
assert os.path.isfile(EXCEL_PATH), f"CLEAN6 workbook not found: {EXCEL_PATH}"
os.makedirs(RAW_LOGS_DIR, exist_ok=True)

def resolve_image_path(base_dir, batch_cat, filename):
    filename = str(filename).strip()
    batch_cat = str(batch_cat or "").strip()
    candidates = [
        os.path.join(base_dir, "images", batch_cat, filename),
        os.path.join(base_dir, batch_cat, filename),
        os.path.join(base_dir, "images", filename),
        os.path.join(base_dir, filename),
    ]
    for candidate in candidates:
        if os.path.isfile(candidate):
            return candidate
    for root, _, files in os.walk(base_dir):
        by_lower = {name.lower(): name for name in files}
        match = by_lower.get(filename.lower())
        if match:
            return os.path.join(root, match)
    raise FileNotFoundError(f"Benchmark image not found: {filename}")

wb_check = openpyxl.load_workbook(EXCEL_PATH, read_only=True, data_only=True)
required_tabs = {"Master_40_Image_Audit", "Primitive_Long_Table", "Summary_KPI_Benchmarks"}
missing_tabs = required_tabs.difference(wb_check.sheetnames)
assert not missing_tabs, f"Workbook is missing required tabs: {sorted(missing_tabs)}"
ws_check = wb_check["Master_40_Image_Audit"]
header_map_check = {str(ws_check.cell(3, c).value).strip(): c for c in range(1, ws_check.max_column + 1) if ws_check.cell(3, c).value}
for required_header in ("image_id", "image_filename", "batch"):
    assert required_header in header_map_check, f"Missing Master sheet header: {required_header}"
image_records = []
for row in range(4, 44):
    filename = ws_check.cell(row, header_map_check["image_filename"]).value
    if filename:
        batch = ws_check.cell(row, header_map_check["batch"]).value
        image_records.append((row, str(filename).strip(), str(batch or "").strip()))
assert len(image_records) == 40, f"Expected 40 benchmark images in Master_40_Image_Audit; found {len(image_records)}"
resolved_paths = {filename: resolve_image_path(BASE_DIR, batch, filename) for _, filename, batch in image_records}
assert len(resolved_paths) == 40 and all(os.path.isfile(p) for p in resolved_paths.values()), "One or more benchmark image paths failed verification"
wb_check.close()

# Colab Secrets only; never display keys, prefixes, or exception details that could contain credentials.
try:
    GOOGLE_API_KEY = userdata.get("GEMINI_API_KEY").strip()
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY").strip()
except Exception:
    raise RuntimeError("Unable to load required Colab Secrets. Check GEMINI_API_KEY and OPENROUTER_API_KEY in Secrets.") from None
if not GOOGLE_API_KEY or not OPENROUTER_API_KEY:
    raise ValueError("Both required Colab Secrets must be non-empty.")
print(f"Verified {len(resolved_paths)}/40 images; workbook tabs: {', '.join(sorted(required_tabs))}. Secrets loaded securely.")


In [ ]:
# Full End-to-End Benchmark Runner: Predictions + Accuracy + Telemetry (4 primitives)
import os, re, json, time, base64
import requests
import openpyxl
from datetime import datetime, timezone

# Set False to update prediction/accuracy fields as well as telemetry.
LATENCY_AND_SPATIAL_CONF_ONLY = False
START_IMAGE = 1
END_IMAGE = 40
GOOGLE_ACTIVE_ENDPOINTS = ["gemini-flash-lite-latest", "gemini-flash-latest", "gemini-3.1-pro-preview"]
JEV_MODEL = "qwen/qwen2.5-vl-72b-instruct"
GOOGLE_MODEL_PRICING = {
    "gemini-flash-lite-latest": {"input": 0.15, "output": 0.60},
    "gemini-flash-latest": {"input": 0.15, "output": 0.60},
    "gemini-3.1-pro-preview": {"input": 0.15, "output": 0.60},
}  # USD per 1M tokens; revise to current contracted rates as needed.

PRIMITIVE_ORDER = ["boolean", "choice", "score", "spatial"]
CHOICE_CLASSES = ["fungal_or_pest_pathology", "drought_curled_high_fire_fuel", "healthy_normal_senescence", "invalid_non_foliage", "nutrient_deficiency_chlorosis"]
SPATIAL_CLASSES = ["top_left", "top", "top_right", "bottom_left", "center", "bottom_right", "bottom", "none"]
PRIMITIVE_PROMPTS = {
    "boolean": "Determine whether active foliar pathology or plant stress is present on target crop foliage. Return JSON {\"pred\": true/false, \"conf\": 0..1}.",
    "choice": f"Choose exactly one class: {CHOICE_CLASSES}. Return JSON {{\"pred\": \"class\", \"conf\": 0..1}}.",
    "score": "Rate foliar desiccation severity from 1 (minimal) to 4 (severe). Return JSON {\"pred\": integer 1..4, \"conf\": 0..1}.",
    "spatial": f"Locate primary symptomatic foliage. Choose one quadrant label {SPATIAL_CLASSES} and box [ymin,xmin,ymax,xmax] scaled 0..1000, or null box if none. Return JSON {{\"pred\": \"label\", \"conf\": 0..1, \"box\": [0,0,0,0]}}.",
}

def _parse_json(text):
    text = re.sub(r"```(?:json)?|```", "", str(text or ""), flags=re.I).strip()
    match = re.search(r"\{.*\}", text, flags=re.S)
    return json.loads(match.group(0)) if match else {}

def _encode_image(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("ascii")

def _google_call(image_b64, note, primitive):
    prompt = f"Scout note: {note}\n{PRIMITIVE_PROMPTS[primitive]} Respond with JSON only."
    body = {"contents":[{"parts":[{"text":prompt},{"inline_data":{"mime_type":"image/jpeg","data":image_b64}}]}], "generationConfig":{"temperature":0,"responseMimeType":"application/json"}}
    last = None
    for model in GOOGLE_ACTIVE_ENDPOINTS:
        for attempt in range(3):
            t0=time.perf_counter()
            try:
                url=f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={GOOGLE_API_KEY}"
                response=requests.post(url,json=body,timeout=90)
                elapsed=round((time.perf_counter()-t0)*1000,1)
                data=response.json()
                usage=data.get("usageMetadata",{})
                in_tok=int(usage.get("promptTokenCount",0) or 0); out_tok=int(usage.get("candidatesTokenCount",0) or 0)
                text=data.get("candidates",[{}])[0].get("content",{}).get("parts",[{}])[0].get("text","")
                parsed=_parse_json(text) if response.ok else {}
                price=GOOGLE_MODEL_PRICING.get(model)
                if price is None: raise ValueError(f"Missing input/output rates for {model}")
                cost=(in_tok*price["input"]+out_tok*price["output"])/1_000_000
                if response.status_code in (429,503) and attempt<2:
                    time.sleep(2*(attempt+1)); continue
                return {"parsed":parsed,"latency":elapsed,"input_tokens":in_tok,"output_tokens":out_tok,"tokens":in_tok+out_tok,"cost":cost,"status":response.status_code,"model":model}
            except Exception as exc:
                last=exc
                if attempt<2: time.sleep(1)
    return {"parsed":{},"latency":0.0,"input_tokens":0,"output_tokens":0,"tokens":0,"cost":0.0,"status":500,"model":GOOGLE_ACTIVE_ENDPOINTS[-1]}

def _jev_call(image_b64, note, primitive):
    body={"model":JEV_MODEL,"temperature":0,"messages":[{"role":"user","content":[{"type":"text","text":f"You are TypeSafe Jev. Scout note: {note}\n{PRIMITIVE_PROMPTS[primitive]} Respond with JSON only."},{"type":"image_url","image_url":{"url":f"data:image/jpeg;base64,{image_b64}"}}]}],"usage":{"include":True}}
    t0=time.perf_counter()
    try:
        response=requests.post("https://openrouter.ai/api/v1/chat/completions",headers={"Authorization":f"Bearer {OPENROUTER_API_KEY}","Content-Type":"application/json"},json=body,timeout=120)
        elapsed=round((time.perf_counter()-t0)*1000,1); data=response.json(); usage=data.get("usage",{})
        text=data.get("choices",[{}])[0].get("message",{}).get("content","")
        return {"parsed":_parse_json(text) if response.ok else {},"latency":elapsed,"input_tokens":int(usage.get("prompt_tokens",0) or 0),"output_tokens":int(usage.get("completion_tokens",0) or 0),"tokens":int(usage.get("total_tokens",0) or 0),"cost":float(usage.get("cost") or 0.0),"status":response.status_code,"model":data.get("model",JEV_MODEL)}
    except Exception:
        return {"parsed":{},"latency":round((time.perf_counter()-t0)*1000,1),"input_tokens":0,"output_tokens":0,"tokens":0,"cost":0.0,"status":500,"model":JEV_MODEL}

def _canon_bool(value):
    return str(value).strip().lower() in ("true","1","yes")
def _canon_spatial(value):
    v=str(value or "").strip().lower().replace("-","_").replace(" ","_")
    return {"top_center":"top","bottom_center":"bottom","center_full":"center","none":"none"}.get(v,v)
def _match(primitive, prediction, truth):
    if prediction is None or truth is None: return None
    if primitive=="boolean": return int(_canon_bool(prediction)==_canon_bool(truth))
    if primitive=="spatial": return int(_canon_spatial(prediction)==_canon_spatial(truth))
    if primitive=="score":
        try: return int(int(prediction)==int(truth))
        except (ValueError,TypeError): return 0
    return int(str(prediction).strip().lower()==str(truth).strip().lower())

def _set(ws,row,headers,name,value):
    col=headers.get(name)
    if col: ws.cell(row,col,value)

def _find_header_map(ws,row=3):
    return {str(ws.cell(row,c).value).strip():c for c in range(1,ws.max_column+1) if ws.cell(row,c).value is not None}

if not (1 <= START_IMAGE <= END_IMAGE <= 40): raise ValueError("START_IMAGE and END_IMAGE must be within 1..40")
wb_run=openpyxl.load_workbook(EXCEL_PATH)
ws=wb_run["Master_40_Image_Audit"]; master_h=_find_header_map(ws)
ws_long=wb_run["Primitive_Long_Table"]; long_h=_find_header_map(ws_long)
long_labels={"boolean":"Boolean Diagnosis","choice":"Categorical Class (5-Way)","score":"Desiccation Score (1-4)","spatial":"Spatial Quadrant / Box"}
long_rows={}
for r in range(4,ws_long.max_row+1):
    key=(str(ws_long.cell(r,long_h["image_id"]).value).strip(),str(ws_long.cell(r,long_h["primitive_name"]).value).strip())
    long_rows[key]=r
master_required={"image_id","image_filename","batch","scout_note"}
assert master_required.issubset(master_h), f"Master headers missing: {sorted(master_required-set(master_h))}"
for img_num in range(START_IMAGE,END_IMAGE+1):
    row=img_num+3; image_id=ws.cell(row,master_h["image_id"]).value
    filename=str(ws.cell(row,master_h["image_filename"]).value).strip(); batch=ws.cell(row,master_h["batch"]).value
    note=str(ws.cell(row,master_h["scout_note"]).value or "")
    image_path=resolved_paths.get(filename) or resolve_image_path(BASE_DIR,batch,filename)
    image_b64=_encode_image(image_path)
    if LATENCY_AND_SPATIAL_CONF_ONLY:
        results={p:{"google":{"parsed":{},"latency":0,"tokens":0,"cost":0,"status":0},"jev":{"parsed":{},"latency":0,"tokens":0,"cost":0,"status":0}} for p in PRIMITIVE_ORDER}
        for p in PRIMITIVE_ORDER:
            results[p]["google"]=_google_call(image_b64,note,p); results[p]["jev"]=_jev_call(image_b64,note,p)
    else:
        results={p:{"google":_google_call(image_b64,note,p),"jev":_jev_call(image_b64,note,p)} for p in PRIMITIVE_ORDER}
    # Write prediction and accuracy data only when explicitly running full benchmark mode.
    if not LATENCY_AND_SPATIAL_CONF_ONLY:
        field_map={"boolean":("G_bool_pred","G_bool_conf","G_bool_match","Jev_bool_pred","Jev_bool_conf","Jev_bool_match","true_boolean_pathology"),"choice":("G_choice_pred","G_choice_conf","G_choice_match","Jev_choice_pred","Jev_choice_conf","Jev_choice_match","true_choice_class"),"score":("G_score_pred","G_score_conf","G_score_exact_match","Jev_score_pred","Jev_score_conf","Jev_score_exact_match","true_score_desiccation_1_to_4"),"spatial":("G_spatial_pred","G_spatial_conf","G_spatial_match","Jev_spatial_pred","Jev_spatial_conf","Jev_spatial_match","true_spatial_quadrant")}
        for p,(gpred,gconf,gmatch,jpred,jconf,jmatch,truth_col) in field_map.items():
            truth=ws.cell(row,master_h[truth_col]).value if truth_col in master_h else None
            gp=results[p]["google"]["parsed"].get("pred"); jp=results[p]["jev"]["parsed"].get("pred")
            g_conf=results[p]["google"]["parsed"].get("conf"); j_conf=results[p]["jev"]["parsed"].get("conf")
            _set(ws,row,master_h,gpred,gp); _set(ws,row,master_h,gconf,g_conf); _set(ws,row,master_h,gmatch,_match(p,gp,truth))
            _set(ws,row,master_h,jpred,jp); _set(ws,row,master_h,jconf,j_conf); _set(ws,row,master_h,jmatch,_match(p,jp,truth))
            if p=="boolean":
                _set(ws,row,master_h,"G_bool_prob_true",(float(g_conf) if _canon_bool(gp) else 1.0-float(g_conf)) if g_conf is not None else None)
                _set(ws,row,master_h,"Jev_bool_prob_true",(float(j_conf) if _canon_bool(jp) else 1.0-float(j_conf)) if j_conf is not None else None)
            if p=="spatial": _set(ws,row,master_h,"Jev_box_coords",json.dumps(results[p]["jev"]["parsed"].get("box")))
    # CLEAN6 fixed telemetry layout (Master columns AM:BD); confidence-only mode preserves prediction/accuracy columns.
    _set(ws,row,master_h,"G_spatial_conf",results["spatial"]["google"]["parsed"].get("conf"))
    _set(ws,row,master_h,"Jev_spatial_conf",results["spatial"]["jev"]["parsed"].get("conf"))
    g_lat=[results[p]["google"]["latency"] for p in PRIMITIVE_ORDER]; j_lat=[results[p]["jev"]["latency"] for p in PRIMITIVE_ORDER]
    for c,v in zip(range(41,45),g_lat): ws.cell(row,c,v)
    _set(ws,row,master_h,"G_total_lat_ms",sum(g_lat))
    for c,v in zip(range(46,50),j_lat): ws.cell(row,c,v)
    _set(ws,row,master_h,"Jev_total_lat_ms",sum(j_lat))
    gs=[results[p]["google"]["status"] for p in PRIMITIVE_ORDER]; js=[results[p]["jev"]["status"] for p in PRIMITIVE_ORDER]
    ws.cell(row,51,max(gs)); ws.cell(row,52,max(js))
    gt=sum(results[p]["google"]["tokens"] for p in PRIMITIVE_ORDER); jt=sum(results[p]["jev"]["tokens"] for p in PRIMITIVE_ORDER)
    ws.cell(row,53,gt); ws.cell(row,54,jt)
    ws.cell(row,55,round(sum(results[p]["google"]["cost"] for p in PRIMITIVE_ORDER),8)); ws.cell(row,56,round(sum(results[p]["jev"]["cost"] for p in PRIMITIVE_ORDER),8))
    _set(ws,row,master_h,"G_api_success",int(all(s==200 for s in gs))); _set(ws,row,master_h,"Jev_api_success",int(all(s==200 for s in js)))
    for p in PRIMITIVE_ORDER:
        lr=long_rows.get((str(image_id).strip(),long_labels[p]))
        if lr:
            _set(ws_long,lr,long_h,"G_latency_ms",results[p]["google"]["latency"]); _set(ws_long,lr,long_h,"Jev_latency_ms",results[p]["jev"]["latency"])
            if not LATENCY_AND_SPATIAL_CONF_ONLY:
                gp=results[p]["google"]["parsed"].get("pred"); jp=results[p]["jev"]["parsed"].get("pred")
                truth=ws_long.cell(lr,long_h["ground_truth"]).value
                _set(ws_long,lr,long_h,"G_prediction",gp); _set(ws_long,lr,long_h,"Jev_prediction",jp)
                _set(ws_long,lr,long_h,"G_confidence_or_exp",results[p]["google"]["parsed"].get("conf")); _set(ws_long,lr,long_h,"Jev_confidence_or_exp",results[p]["jev"]["parsed"].get("conf"))
                _set(ws_long,lr,long_h,"G_match",_match(p,gp,truth)); _set(ws_long,lr,long_h,"Jev_match",_match(p,jp,truth))
                if p=="spatial": _set(ws_long,lr,long_h,"Jev_box_coords_spatial",json.dumps(results[p]["jev"]["parsed"].get("box")))
            if p=="spatial":
                _set(ws_long,lr,long_h,"G_confidence_or_exp",results[p]["google"]["parsed"].get("conf")); _set(ws_long,lr,long_h,"Jev_confidence_or_exp",results[p]["jev"]["parsed"].get("conf"))
    wb_run.save(EXCEL_PATH)
    print(f"[{img_num:02d}/40] {filename}: checkpoint saved; spatial confidence, latency, status, tokens, cost recorded.")
print(f"Completed images {START_IMAGE}–{END_IMAGE}. Workbook checkpointed at {EXCEL_PATH}.")


In [ ]:
# Configurable Range Runner: Spatial Confidence, Latency & Exact Cost Telemetry
import os, re, json, time, base64
import requests
import openpyxl
from google.colab import userdata

START_IMAGE = 1
END_IMAGE = 40
# True preserves verified predictions/accuracy in columns K:AK; False also writes refreshed predictions and accuracy.
LATENCY_AND_SPATIAL_CONF_ONLY = True
GOOGLE_FREE_TIER = False
BASE_DIR = os.environ.get("AGRITECH_BASE_DIR")
if not BASE_DIR:
    raise RuntimeError("Set AGRITECH_BASE_DIR to the benchmark project directory before running this cell.")
EXCEL_PATH = f"{BASE_DIR}/Project_AgriTech_Decision_Audit_CLEAN6.xlsx"
GOOGLE_API_KEY = userdata.get("GEMINI_API_KEY").strip()
OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY").strip()
GOOGLE_MODEL = "gemini-flash-lite-latest"
JEV_MODEL = "qwen/qwen2.5-vl-72b-instruct"
# USD per 1M tokens, by input/output as applicable. Update rates if provider pricing changes.
GOOGLE_MODEL_PRICING = {GOOGLE_MODEL: {"input": 0.15, "output": 0.60}}
GOOGLE_FREE_TIER = bool(GOOGLE_FREE_TIER)
if GOOGLE_FREE_TIER:
    GOOGLE_MODEL_PRICING[GOOGLE_MODEL] = {"input": 0.0, "output": 0.0}

PRIMITIVES = ["boolean", "choice", "score", "spatial"]
PROMPTS = {
 "boolean": 'Is active foliar disease or pest damage present? JSON only: {"pred":true/false,"conf":0.0}.',
 "choice": 'Choose one of fungal_or_pest_pathology, drought_curled_high_fire_fuel, healthy_normal_senescence, invalid_non_foliage, nutrient_deficiency_chlorosis. JSON only: {"pred":"label","conf":0.0}.',
 "score": 'Score foliar desiccation from 1 to 4. JSON only: {"pred":1,"conf":0.0}.',
 "spatial": 'Choose quadrant top_left, top, top_right, bottom_left, center, bottom_right, bottom, or none; provide [ymin,xmin,ymax,xmax] normalized 0..1000 or null. JSON only: {"pred":"label","conf":0.0,"box":null}.'
}
def parse_json_safely(text):
    clean=re.sub(r"```(?:json)?|```", "", str(text or ""), flags=re.I).strip()
    match=re.search(r"\{.*\}",clean,flags=re.S)
    try: return json.loads(match.group(0)) if match else {}
    except (ValueError,TypeError): return {}
def encode_image_b64(img_path):
    with open(img_path,"rb") as f: return base64.b64encode(f.read()).decode("ascii")
def _provider_result(parsed,latency,tokens_in,tokens_out,status,cost,model):
    return {"parsed":parsed,"latency":latency,"tokens_in":tokens_in,"tokens_out":tokens_out,"tokens":tokens_in+tokens_out,"status":status,"cost":cost,"model":model}
def call_google_isolated(img_b64, scout_note, primitive_type, max_retries=3):
    prompt=f"Scout context: {scout_note}\n{PROMPTS[primitive_type]}"
    body={"contents":[{"parts":[{"text":prompt},{"inline_data":{"mime_type":"image/jpeg","data":img_b64}}]}],"generationConfig":{"temperature":0,"responseMimeType":"application/json"}}
    last=_provider_result({},0,0,0,500,0,GOOGLE_MODEL)
    for attempt in range(max_retries):
        t0=time.perf_counter()
        try:
            url=f"https://generativelanguage.googleapis.com/v1beta/models/{GOOGLE_MODEL}:generateContent?key={GOOGLE_API_KEY}"
            response=requests.post(url,json=body,timeout=90); data=response.json(); usage=data.get("usageMetadata",{})
            ins=int(usage.get("promptTokenCount",0) or 0); outs=int(usage.get("candidatesTokenCount",0) or 0)
            rates=GOOGLE_MODEL_PRICING[GOOGLE_MODEL]; cost=(ins*rates["input"]+outs*rates["output"])/1_000_000
            text=data.get("candidates",[{}])[0].get("content",{}).get("parts",[{}])[0].get("text","")
            last=_provider_result(parse_json_safely(text) if response.ok else {},round((time.perf_counter()-t0)*1000,1),ins,outs,response.status_code,cost,GOOGLE_MODEL)
            if response.status_code==429 and attempt<max_retries-1: time.sleep(2*(attempt+1)); continue
            return last
        except Exception:
            last=_provider_result({},round((time.perf_counter()-t0)*1000,1),0,0,500,0,GOOGLE_MODEL)
            if attempt<max_retries-1: time.sleep(1)
    return last
def call_jev_isolated(img_b64, scout_note, primitive_type, max_retries=3):
    body={"model":JEV_MODEL,"temperature":0,"messages":[{"role":"user","content":[{"type":"text","text":f"Scout context: {scout_note}\n{PROMPTS[primitive_type]}"},{"type":"image_url","image_url":{"url":f"data:image/jpeg;base64,{img_b64}"}}]}],"usage":{"include":True}}
    headers={"Authorization":f"Bearer {OPENROUTER_API_KEY}","Content-Type":"application/json"}; last=_provider_result({},0,0,0,500,0,JEV_MODEL)
    for attempt in range(max_retries):
        t0=time.perf_counter()
        try:
            response=requests.post("https://openrouter.ai/api/v1/chat/completions",headers=headers,json=body,timeout=120); data=response.json(); usage=data.get("usage",{})
            ins=int(usage.get("prompt_tokens",0) or 0); outs=int(usage.get("completion_tokens",0) or 0)
            total=int(usage.get("total_tokens",ins+outs) or (ins+outs)); cost=float(usage.get("cost") or 0.0)
            text=data.get("choices",[{}])[0].get("message",{}).get("content","")
            last=_provider_result(parse_json_safely(text) if response.ok else {},round((time.perf_counter()-t0)*1000,1),ins,outs,response.status_code,cost,data.get("model",JEV_MODEL))
            last["tokens"]=total
            if response.status_code==429 and attempt<max_retries-1: time.sleep(3*(attempt+1)); continue
            return last
        except Exception:
            last=_provider_result({},round((time.perf_counter()-t0)*1000,1),0,0,500,0,JEV_MODEL)
            if attempt<max_retries-1: time.sleep(1)
    return last
def resolve_image_path(base_dir,batch_cat,filename):
    filename=str(filename).strip(); batch_cat=str(batch_cat or "").strip()
    for candidate in [os.path.join(base_dir,"images",batch_cat,filename),os.path.join(base_dir,batch_cat,filename),os.path.join(base_dir,"images",filename),os.path.join(base_dir,filename)]:
        if os.path.isfile(candidate): return candidate
    for root,_,files in os.walk(base_dir):
        found=next((f for f in files if f.lower()==filename.lower()),None)
        if found:return os.path.join(root,found)
    raise FileNotFoundError(f"Benchmark image not found: {filename}")
def _header_map(ws): return {str(ws.cell(3,c).value).strip():c for c in range(1,ws.max_column+1) if ws.cell(3,c).value is not None}
def _set(ws,row,hdr,name,val):
    if name in hdr: ws.cell(row,hdr[name],val)
def _canonical(primitive,pred):
    if primitive=="boolean": return str(pred).strip().lower() in ("true","1","yes")
    if primitive=="spatial": return str(pred or "").lower().replace("-","_").replace(" ","_")
    if primitive=="score":
        try:return int(pred)
        except (TypeError,ValueError):return pred
    return str(pred or "").strip().lower()
def _accuracy(primitive,pred,truth):
    return None if pred is None or truth is None else int(_canonical(primitive,pred)==_canonical(primitive,truth))
if not (1<=START_IMAGE<=END_IMAGE<=40): raise ValueError("START_IMAGE and END_IMAGE must be within 1..40")
wb=openpyxl.load_workbook(EXCEL_PATH); ws_master=wb["Master_40_Image_Audit"]; master_h=_header_map(ws_master); ws_long=wb["Primitive_Long_Table"]; long_h=_header_map(ws_long)
long_labels={"boolean":"Boolean Diagnosis","choice":"Categorical Class (5-Way)","score":"Desiccation Score (1-4)","spatial":"Spatial Quadrant / Box"}
long_rows={}
for r in range(4,ws_long.max_row+1): long_rows[(str(ws_long.cell(r,long_h["image_id"]).value).strip(),str(ws_long.cell(r,long_h["primitive_name"]).value).strip())]=r
field_map={"boolean":("G_bool_pred","G_bool_conf","G_bool_match","Jev_bool_pred","Jev_bool_conf","Jev_bool_match","true_boolean_pathology"),"choice":("G_choice_pred","G_choice_conf","G_choice_match","Jev_choice_pred","Jev_choice_conf","Jev_choice_match","true_choice_class"),"score":("G_score_pred","G_score_conf","G_score_exact_match","Jev_score_pred","Jev_score_conf","Jev_score_exact_match","true_score_desiccation_1_to_4"),"spatial":("G_spatial_pred","G_spatial_conf","G_spatial_match","Jev_spatial_pred","Jev_spatial_conf","Jev_spatial_match","true_spatial_quadrant")}
for img_num in range(START_IMAGE,END_IMAGE+1):
    row=img_num+3; image_id=ws_master.cell(row,master_h["image_id"]).value; filename=str(ws_master.cell(row,master_h["image_filename"]).value).strip(); batch=ws_master.cell(row,master_h["batch"]).value; note=str(ws_master.cell(row,master_h["scout_note"]).value or "")
    img_b64=encode_image_b64(resolve_image_path(BASE_DIR,batch,filename)); results={}
    for primitive in PRIMITIVES:
        results[primitive]={"google":call_google_isolated(img_b64,note,primitive),"jev":call_jev_isolated(img_b64,note,primitive)}
    # Required CLEAN6 Master mappings: AM/AN=39/40; AO:AR=41:44; AT:AW=46:49; AY:AZ=51:52; BA:BB=53:54; BC:BD=55:56.
    ws_master.cell(row,39,results["spatial"]["google"]["parsed"].get("conf")); ws_master.cell(row,40,results["spatial"]["jev"]["parsed"].get("conf"))
    for col,prim in zip(range(41,45),PRIMITIVES): ws_master.cell(row,col,results[prim]["google"]["latency"])
    _set(ws_master,row,master_h,"G_total_lat_ms",sum(results[p]["google"]["latency"] for p in PRIMITIVES))
    for col,prim in zip(range(46,50),PRIMITIVES): ws_master.cell(row,col,results[prim]["jev"]["latency"])
    _set(ws_master,row,master_h,"Jev_total_lat_ms",sum(results[p]["jev"]["latency"] for p in PRIMITIVES))
    ws_master.cell(row,51,max(results[p]["google"]["status"] for p in PRIMITIVES)); ws_master.cell(row,52,max(results[p]["jev"]["status"] for p in PRIMITIVES))
    ws_master.cell(row,53,sum(results[p]["google"]["tokens"] for p in PRIMITIVES)); ws_master.cell(row,54,sum(results[p]["jev"]["tokens"] for p in PRIMITIVES))
    ws_master.cell(row,55,round(sum(results[p]["google"]["cost"] for p in PRIMITIVES),8)); ws_master.cell(row,56,round(sum(results[p]["jev"]["cost"] for p in PRIMITIVES),8))
    if not LATENCY_AND_SPATIAL_CONF_ONLY:
        for p,fields in field_map.items():
            gp=results[p]["google"]["parsed"].get("pred"); jp=results[p]["jev"]["parsed"].get("pred"); truth=ws_master.cell(row,master_h[fields[6]]).value
            _set(ws_master,row,master_h,fields[0],gp); _set(ws_master,row,master_h,fields[1],results[p]["google"]["parsed"].get("conf")); _set(ws_master,row,master_h,fields[2],_accuracy(p,gp,truth))
            _set(ws_master,row,master_h,fields[3],jp); _set(ws_master,row,master_h,fields[4],results[p]["jev"]["parsed"].get("conf")); _set(ws_master,row,master_h,fields[5],_accuracy(p,jp,truth))
        _set(ws_master,row,master_h,"Jev_box_coords",json.dumps(results["spatial"]["jev"]["parsed"].get("box")))
    for p in PRIMITIVES:
        lr=long_rows.get((str(image_id).strip(),long_labels[p]))
        if not lr: continue
        ws_long.cell(lr,18,results[p]["google"]["latency"]); ws_long.cell(lr,19,results[p]["jev"]["latency"])
        if not LATENCY_AND_SPATIAL_CONF_ONLY:
            gp=results[p]["google"]["parsed"].get("pred"); jp=results[p]["jev"]["parsed"].get("pred"); truth=ws_long.cell(lr,long_h["ground_truth"]).value
            _set(ws_long,lr,long_h,"G_prediction",gp); _set(ws_long,lr,long_h,"Jev_prediction",jp); _set(ws_long,lr,long_h,"G_confidence_or_exp",results[p]["google"]["parsed"].get("conf")); _set(ws_long,lr,long_h,"Jev_confidence_or_exp",results[p]["jev"]["parsed"].get("conf")); _set(ws_long,lr,long_h,"G_match",_accuracy(p,gp,truth)); _set(ws_long,lr,long_h,"Jev_match",_accuracy(p,jp,truth))
            if p=="spatial": _set(ws_long,lr,long_h,"Jev_box_coords_spatial",json.dumps(results[p]["jev"]["parsed"].get("box")))
        if p=="spatial":
            ws_long.cell(lr,10,results[p]["google"]["parsed"].get("conf")); ws_long.cell(lr,13,results[p]["jev"]["parsed"].get("conf"))
    wb.save(EXCEL_PATH)
    print(f"[{img_num:02d}/40] {filename}: confidence/latency/HTTP/token/cost telemetry saved.")
print(f"Completed range {START_IMAGE}..{END_IMAGE}; workbook saved to {EXCEL_PATH}.")
